In [1]:
# NB04b_override_simulation_independent.ipynb

# %% [markdown]
# # NB04b – Override Simulation under Alternative Data-Generating Mechanisms
# **Purpose:** Replace the single naive simulation with three explicit DGMs,
# so that the sensitivity of all downstream findings to the simulation design
# can be assessed directly (addressing the circularity critique).
#
# **Three data-generating mechanisms:**
#   naive : upgrade probability driven by system_grade (== pd_system quantile).
#           Reproduces the original design. Serves as a WARNING baseline:
#           default risk and upgrade probability both rise with pd_system,
#           so the upgrade group mechanically over-samples high-risk firms.
#   b1    : risk-neutral upgrades. Upgrade probability driven by Attr43,
#           which is near-orthogonal to BOTH pd_system and default. The
#           induced selection is approximately random w.r.t. risk, so any
#           residual grade_diff effect cannot be a mechanical artifact.
#   b2    : risk-informed, pd-independent upgrades. Upgrade probability driven
#           by Attr29 (firm size), weakly correlated with default but not with
#           pd_system. Tests whether an override signal that the system model
#           did NOT already capture survives without pd_system circularity.
#
# **Sign convention (unified with the code, corrected in the paper):**
#   grade_diff = final_ordinal - system_ordinal
#   NEGATIVE grade_diff = UPGRADE (lower ordinal = better grade)
#   POSITIVE grade_diff = DOWNGRADE
#
# **Input:**  `data/processed/graded_data.parquet`
# **Output:**
#   - `data/processed/override_data_naive.parquet`
#   - `data/processed/override_data_b1.parquet`
#   - `data/processed/override_data_b2.parquet`
#   - `results/tables/04b_dgm_comparison.csv`

# %%
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROC_DIR  = "../data/processed/"
TABLE_DIR = "../results/tables/"

graded = pd.read_parquet(PROC_DIR + "graded_data.parquet")

GRADE_LABELS = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]
GRADE_ORDER  = {g: i for i, g in enumerate(GRADE_LABELS)}
INV_GRADE    = {v: k for k, v in GRADE_ORDER.items()}

# Grade-based upgrade probabilities (naive DGM only)
GRADE_UPGRADE_PROB = {
    "AAA": 0.10, "AA": 0.15, "A": 0.20, "BBB": 0.35,
    "BB": 0.50, "B": 0.70, "CCC": 0.80,
}

# %% [markdown]
# ## 1. Unified override simulation engine
# The only thing that differs across DGMs is how the per-firm upgrade
# probability is computed. Everything downstream (magnitude, direction,
# grade_diff) is identical, so the DGMs are strictly comparable.

# %%
def _upgrade_prob_naive(df):
    """Upgrade prob is a function of system_grade (== pd_system quantile)."""
    return df["system_grade"].map(GRADE_UPGRADE_PROB).values

def _upgrade_prob_from_driver(df, driver_col, p_low=0.10, p_high=0.80):
    """
    Upgrade prob is a monotone function of a driver variable that is
    (near-)independent of pd_system. Firms in higher quantiles of the driver
    receive higher upgrade probability, linearly interpolated in [p_low, p_high].
    This mirrors the *spread* of the naive probabilities (0.10-0.80) but ties
    them to a pd-independent signal instead of to system risk.
    """
    x = df[driver_col].astype(float)
    # rank-transform to [0, 1]; robust to scale and outliers, NaN-safe
    r = x.rank(pct=True, na_option="keep").values
    r = np.where(np.isnan(r), 0.5, r)          # missing -> neutral prob
    return p_low + (p_high - p_low) * r

def simulate_override(df, dgm="naive", driver_col=None,
                      override_rate=0.35, seed=None,
                      notch1_share=0.70):
    """
    Simulate evaluator grade adjustments under a chosen DGM.

    Parameters
    ----------
    dgm           : 'naive' | 'b1' | 'b2'
    driver_col    : required for b1/b2 (e.g. 'Attr43', 'Attr29')
    override_rate : share of firms receiving any override
    seed          : RNG seed (set per replication in NB04c)
    notch1_share  : share of overrides that move exactly 1 notch

    Returns
    -------
    DataFrame with override_flag, override_direction, final_grade,
    final_ordinal, grade_diff.
    """
    rng = np.random.default_rng(seed)
    df  = df.copy()
    n   = len(df)

    if dgm == "naive":
        upgrade_prob = _upgrade_prob_naive(df)
    elif dgm in ("b1", "b2"):
        if driver_col is None:
            raise ValueError(f"{dgm} requires driver_col")
        upgrade_prob = _upgrade_prob_from_driver(df, driver_col)
    else:
        raise ValueError(f"unknown dgm: {dgm}")

    override_mask = rng.random(n) < override_rate
    upgrade_flag  = rng.random(n) < upgrade_prob
    notch         = np.where(rng.random(n) < notch1_share, 1, 2)

    # NEGATIVE delta = upgrade (lower ordinal = better grade)
    delta = np.where(upgrade_flag, -notch, notch)

    new_ordinal = df["grade_ordinal"].values.copy().astype(int)
    new_ordinal[override_mask] = (
        new_ordinal[override_mask] + delta[override_mask]
    ).clip(0, 6)

    direction = np.full(n, "none", dtype=object)
    applied_delta = new_ordinal - df["grade_ordinal"].values
    direction[(override_mask) & (applied_delta < 0)] = "upgrade"
    direction[(override_mask) & (applied_delta > 0)] = "downgrade"

    df["override_flag"]      = override_mask.astype(int)
    df["final_ordinal"]      = new_ordinal
    df["final_grade"]        = [INV_GRADE[o] for o in new_ordinal]
    df["grade_diff"]         = applied_delta          # negative = upgrade
    df["override_direction"] = direction
    return df

# %% [markdown]
# ## 2. Run the three DGMs once (point estimates)
# Distributional inference over many seeds is done in NB04c.

# %%
DGM_CONFIG = {
    "naive": dict(dgm="naive", driver_col=None),
    "b1"   : dict(dgm="b1",    driver_col="Attr43"),   # risk-neutral
    "b2"   : dict(dgm="b2",    driver_col="Attr29"),   # size, pd-independent
}

def summarize(df_sim):
    """Return upgrade/none/downgrade default rates for one simulated frame."""
    g = (df_sim.groupby("override_direction")["default"]
               .agg(n="count", dr="mean")
               .reindex(["none", "upgrade", "downgrade"]))
    return g

rows = []
for name, cfg in DGM_CONFIG.items():
    sim = simulate_override(graded, override_rate=0.35, seed=2024, **cfg)
    g = summarize(sim)
    dr_none = g.loc["none", "dr"]
    dr_up   = g.loc["upgrade", "dr"]
    dr_down = g.loc["downgrade", "dr"]
    rows.append({
        "dgm"           : name,
        "driver"        : cfg["driver_col"] or "system_grade",
        "n_upgrade"     : int(g.loc["upgrade", "n"]),
        "dr_none"       : round(dr_none, 4),
        "dr_upgrade"    : round(dr_up, 4),
        "dr_downgrade"  : round(dr_down, 4),
        "upgrade_minus_none": round(dr_up - dr_none, 4),
    })
    sim.to_parquet(PROC_DIR + f"override_data_{name}.parquet", index=False)

comparison = pd.DataFrame(rows)
comparison.to_csv(TABLE_DIR + "04b_dgm_comparison.csv", index=False)

print("=== DGM comparison (single-seed point estimates) ===")
print("Expectation: the upgrade>none gap SHRINKS from naive -> b1,")
print("and is small under b2, revealing how much of the naive effect")
print("is mechanically induced by pd_system circularity.\n")
print(comparison.to_string(index=False))

=== DGM comparison (single-seed point estimates) ===
Expectation: the upgrade>none gap SHRINKS from naive -> b1,
and is small under b2, revealing how much of the naive effect
is mechanically induced by pd_system circularity.

  dgm       driver  n_upgrade  dr_none  dr_upgrade  dr_downgrade  upgrade_minus_none
naive system_grade       6125   0.0484      0.0802        0.0246              0.0318
   b1       Attr43       6094   0.0500      0.0563        0.0344              0.0063
   b2       Attr29       6224   0.0504      0.0516        0.0359              0.0011
